In [ ]:

# 1. Setup
!pip install -q openai textstat pandas
import pandas as pd
import textstat
from openai import OpenAI

# Reference your existing api_key variable
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=api_key)

# 2. Updated Models & Variation Prompts
models = {
    "GPT-4o": "openai/gpt-4o",
    "Claude-3.5": "anthropic/claude-3.5-sonnet",
    "Gemini-2.5": "google/gemini-2.5-pro-preview" # Updated ID
}

prompts = [
    "The cat sat on the mat.",                  # Grade 1-2
    "How do you bake a simple chocolate cake?", # Grade 4-5
    "Explain why the sky is blue to a child.",  # Grade 5-6
    "What are the main causes of inflation?",   # Grade 8-10
    "Summarize the laws of thermodynamics.",    # Grade 12+
    "Explain quantum tunneling in transistors.", # Academic
]

results = []
print("Processing... ⏳")

for label, model_id in models.items():
    for p in prompts:
        try:
            # Calculate input grade
            p_grade = textstat.flesch_kincaid_grade(p)

            # Get response
            res = client.chat.completions.create(
                model=model_id,
                messages=[{"role": "user", "content": p}]
            )
            out = res.choices[0].message.content

            # Calculate output grade
            o_grade = textstat.flesch_kincaid_grade(out)

            results.append([label, p_grade, o_grade])
        except Exception as e:
            print(f"Error with {label}: {e}")

# 3. Simplified Table for Mobile Screenshot
df = pd.DataFrame(results, columns=["Model", "In", "Out"])

print("\n--- GRADE LEVEL RESULTS ---")
print(df.to_string(index=False))

# Calculate Correlation per model
print("\n--- CORRELATION ---")
for m in df['Model'].unique():
    subset = df[df['Model'] == m]
    corr = subset['In'].corr(subset['Out'])
    print(f"{m}: {corr:.2f}")